# TF-ISF vs BM25 vs Cosine Retrieval Evaluation

**Demo Notebook**: Comprehensive evaluation of three retrieval methods on scientific QA (QASPER dataset).

This notebook loads per-example predictions from an evaluation experiment (n=180, tencent/hy3:free LLM) and computes:
- Pairwise F1 comparisons with bootstrap confidence intervals
- Effect sizes (Cohen's d, Hedges' g) and statistical significance tests
- Hallucination rates and retrieval recall statistics
- Subgroup analysis by document section type
- Kruskal-Wallis variance decomposition
- Reliability assessment of the experiment

**Key Finding**: No significant differences between methods after multiple-comparison correction (Holm-Bonferroni). High hallucination rates (62-72%) indicate the LLM quality is the bottleneck, not the retrieval method choice.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

# Non-Colab packages
_pip('loguru==0.7.2')

In [ ]:
import json
import math
from collections import defaultdict

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Display settings
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-023b95-within-document-term-weighting-for-scien/main/round-2/evaluation-1/demo/mini_demo_data.json"
import os

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'][0]['examples'])} examples")
print(f"Methods: {data['metadata']['methods']}")

## Configuration

All tunable parameters are set to DEMO values (minimal scale). Scale up the `N_BOOTSTRAP` or use more examples to reproduce the full evaluation.

In [ ]:
# Demo configuration (small scale for quick testing)
N_BOOTSTRAP = 100  # DEMO: 100 bootstrap replicates (original: 10_000)
N_EXAMPLES_TO_USE = 3  # DEMO: use only 3 examples (original: all 180)
RNG_SEED = 42
METHODS = ["cosine", "bm25", "tf_isf"]
CI_LEVEL = 0.95

# Original full-scale config (commented out):
# N_BOOTSTRAP = 10_000  # Full evaluation uses 10,000 bootstrap replicates
# N_EXAMPLES_TO_USE = 180  # Full: all examples from iter_1 experiment

print(f"Demo Config: N_BOOTSTRAP={N_BOOTSTRAP}, N_EXAMPLES={N_EXAMPLES_TO_USE}")

## Extract Per-Example Metrics

Extract F1, recall, and hallucination indicators from the demo data. Each example has predictions from all three methods.

In [ ]:
def parse_f1(v) -> float:
    """Parse F1 value, defaulting to 0.0 on error."""
    try:
        return float(v)
    except (TypeError, ValueError):
        return 0.0

examples = data['datasets'][0]['examples'][:N_EXAMPLES_TO_USE]
n = len(examples)

# Initialize per-method arrays
f1 = {m: [] for m in METHODS}
recall = {m: [] for m in METHODS}
gold_types = []

# Extract metrics from examples
for ex in examples:
    gold_types.append(ex.get("metadata_gold_section_type", "Unknown"))
    for m in METHODS:
        f1[m].append(parse_f1(ex.get(f"metadata_f1_{m}", 0)))
        recall[m].append(parse_f1(ex.get(f"metadata_section_recall_{m}", 0)))

# Convert to numpy arrays
for m in METHODS:
    f1[m] = np.array(f1[m])
    recall[m] = np.array(recall[m])

print(f"Extracted metrics from {n} examples")
print(f"Section types: {sorted(set(gold_types))}")

## Descriptive Statistics

Compute mean, std, min, max, and quartiles for F1 and recall per method.

In [ ]:
def describe(arr: np.ndarray) -> dict:
    """Return descriptive statistics for an array."""
    return {
        "n": int(len(arr)),
        "mean": float(arr.mean()),
        "std": float(arr.std(ddof=1) if len(arr) > 1 else 0),
        "min": float(arr.min()),
        "q25": float(np.percentile(arr, 25)),
        "median": float(np.median(arr)),
        "q75": float(np.percentile(arr, 75)),
        "max": float(arr.max()),
    }

desc_stats_f1 = {m: describe(f1[m]) for m in METHODS}
desc_stats_recall = {m: describe(recall[m]) for m in METHODS}

print("\n=== F1 DESCRIPTIVE STATISTICS ===")
for m in METHODS:
    s = desc_stats_f1[m]
    print(f"{m:8s}: mean={s['mean']:.4f} std={s['std']:.4f} median={s['median']:.4f} [min={s['min']:.4f}, max={s['max']:.4f}]")

print("\n=== RECALL DESCRIPTIVE STATISTICS ===")
for m in METHODS:
    s = desc_stats_recall[m]
    print(f"{m:8s}: mean={s['mean']:.4f} std={s['std']:.4f} median={s['median']:.4f} [min={s['min']:.4f}, max={s['max']:.4f}]")

## Bootstrap Confidence Intervals

Compute 95% confidence intervals for each method's mean F1 using percentile bootstrap.

In [ ]:
def bootstrap_ci(
    arr: np.ndarray,
    n_boot: int = N_BOOTSTRAP,
    ci: float = CI_LEVEL,
    rng: np.random.Generator = None,
) -> tuple:
    """Return (mean, lower_ci, upper_ci) via percentile bootstrap."""
    if rng is None:
        rng = np.random.default_rng(RNG_SEED)
    if len(arr) < 2:
        return float(arr.mean()), float(arr.mean()), float(arr.mean())
    means = np.array([rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_boot)])
    alpha = (1 - ci) / 2
    return float(arr.mean()), float(np.percentile(means, 100 * alpha)), float(np.percentile(means, 100 * (1 - alpha)))

rng = np.random.default_rng(RNG_SEED)
method_ci = {}
for m in METHODS:
    mu, lo, hi = bootstrap_ci(f1[m], rng=rng)
    method_ci[m] = (mu, lo, hi)

print("\n=== BOOTSTRAP 95% CONFIDENCE INTERVALS (F1) ===")
for m in METHODS:
    mu, lo, hi = method_ci[m]
    width = hi - lo
    print(f"{m:8s}: F1={mu:.4f} 95%CI=[{lo:.4f}, {hi:.4f}] width={width:.4f}")

## Pairwise Comparisons

Compare methods pairwise using bootstrap difference CIs and Holm-Bonferroni correction for multiple comparisons.

In [ ]:
def bootstrap_diff_ci(
    a: np.ndarray,
    b: np.ndarray,
    n_boot: int = N_BOOTSTRAP,
    ci: float = CI_LEVEL,
    rng: np.random.Generator = None,
) -> tuple:
    """Return (diff, lower, upper, p_value) for unpaired mean difference a - b."""
    if rng is None:
        rng = np.random.default_rng(RNG_SEED)
    if len(a) < 2 or len(b) < 2:
        diff = float(a.mean() - b.mean())
        return diff, diff, diff, 1.0
    diff_obs = a.mean() - b.mean()
    diffs = np.array([
        rng.choice(a, size=len(a), replace=True).mean() -
        rng.choice(b, size=len(b), replace=True).mean()
        for _ in range(n_boot)
    ])
    alpha = (1 - ci) / 2
    lo = float(np.percentile(diffs, 100 * alpha))
    hi = float(np.percentile(diffs, 100 * (1 - alpha)))
    p_val = float(2 * min((diffs >= 0).mean(), (diffs <= 0).mean()))
    return float(diff_obs), lo, hi, p_val

def cohen_d(a: np.ndarray, b: np.ndarray) -> float:
    """Compute Cohen's d effect size."""
    if len(a) < 2 or len(b) < 2:
        return 0.0
    pooled_std = math.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    if pooled_std == 0:
        return 0.0
    return float((a.mean() - b.mean()) / pooled_std)

def hedges_g(a: np.ndarray, b: np.ndarray) -> float:
    """Compute Hedges' g effect size (Cohen's d with correction)."""
    n1, n2 = len(a), len(b)
    if n1 < 2 or n2 < 2:
        return 0.0
    d = cohen_d(a, b)
    df = n1 + n2 - 2
    j = 1 - (3 / (4 * df - 1)) if df > 0 else 1.0
    return float(d * j)

def holm_bonferroni(p_values: list) -> list:
    """Return Holm-Bonferroni corrected p-values."""
    n = len(p_values)
    indexed = sorted(enumerate(p_values), key=lambda x: x[1])
    corrected = [0.0] * n
    prev = 0.0
    for rank, (orig_i, p) in enumerate(indexed):
        adj = min(1.0, max(prev, p * (n - rank)))
        corrected[orig_i] = adj
        prev = adj
    return corrected

In [ ]:
# Perform pairwise comparisons
pairs = [("tf_isf", "cosine"), ("tf_isf", "bm25"), ("bm25", "cosine")]
raw_p_values = []
pair_results_raw = []

for a, b in pairs:
    diff, lo, hi, p = bootstrap_diff_ci(f1[a], f1[b], rng=rng)
    raw_p_values.append(p)
    pair_results_raw.append({
        "pair": f"{a}_vs_{b}",
        "diff": diff,
        "ci_lo": lo,
        "ci_hi": hi,
        "p_raw": p,
        "cohen_d": cohen_d(f1[a], f1[b]),
        "hedges_g": hedges_g(f1[a], f1[b]),
    })

# Apply Holm-Bonferroni correction
corrected_p = holm_bonferroni(raw_p_values)
pairwise_comparisons = []
for i, res in enumerate(pair_results_raw):
    res["p_holm_bonferroni"] = corrected_p[i]
    res["significant_hb_alpha05"] = corrected_p[i] < 0.05
    pairwise_comparisons.append(res)

print("\n=== PAIRWISE COMPARISONS (with Holm-Bonferroni correction) ===")
for res in pairwise_comparisons:
    sig = "SIGNIFICANT" if res["significant_hb_alpha05"] else "not significant"
    print(f"{res['pair']:20s}: diff={res['diff']:+.4f} CI=[{res['ci_lo']:.4f},{res['ci_hi']:.4f}]")
    print(f"  → p_raw={res['p_raw']:.4f}, p_hb={res['p_holm_bonferroni']:.4f} [{sig}], Hedges'g={res['hedges_g']:.4f}")

## Hallucination Analysis

Hallucination: F1 > 0 (model generated something reasonable) but recall = 0 (no relevant section was retrieved). This indicates the LLM is generating plausible text from memory, not from retrieval.

In [ ]:
halluc = {}
for m in METHODS:
    halluc_count = 0
    for i in range(n):
        f1_val = f1[m][i]
        rec_val = recall[m][i]
        if f1_val > 0 and rec_val == 0:
            halluc_count += 1
    rate = halluc_count / n * 100 if n > 0 else 0
    halluc[m] = {"count": halluc_count, "rate_pct": rate}

print("\n=== HALLUCINATION RATES ===")
print("(F1 > 0 despite zero retrieval recall)")
for m in METHODS:
    print(f"{m:8s}: {halluc[m]['count']} hallucinated / {n} total = {halluc[m]['rate_pct']:.1f}%")

## Kruskal-Wallis Test

Non-parametric test for whether F1 distributions differ across the three methods. H statistic and p-value indicate overall variance explained by method choice.

In [ ]:
kw_stat, kw_p = stats.kruskal(f1["cosine"], f1["bm25"], f1["tf_isf"])

# eta-squared approximation for Kruskal-Wallis
k = 3  # number of groups
n_total = n * k
kw_eta2 = (kw_stat - k + 1) / (n_total - k) if n_total > k else 0.0

print("\n=== KRUSKAL-WALLIS TEST ===")
print(f"H statistic = {kw_stat:.4f}")
print(f"p-value = {kw_p:.4f}")
print(f"eta² (variance explained) ≈ {kw_eta2:.4f}")
if kw_p < 0.05:
    print("→ Methods have SIGNIFICANTLY DIFFERENT F1 distributions (p < 0.05)")
else:
    print("→ No significant difference in F1 distributions across methods (p ≥ 0.05)")

## Subgroup Analysis by Section Type

Break down performance by gold section type (Abstract, Method, Result, etc.) to see if method advantage varies by document section.

In [ ]:
unique_types = sorted(set(gold_types))
gold_types_arr = np.array(gold_types)
subgroup_results = {}

print("\n=== SUBGROUP ANALYSIS BY SECTION TYPE ===")
for stype in unique_types:
    mask = gold_types_arr == stype
    n_sub = int(mask.sum())
    if n_sub < 1:
        continue
    sub = {m: f1[m][mask] for m in METHODS}
    sub_means = {m: float(sub[m].mean()) for m in METHODS}
    best_method = max(sub_means, key=sub_means.get)
    subgroup_results[stype] = {"n": n_sub, "means": sub_means, "best_method": best_method}
    print(f"{stype:12s} (n={n_sub}): {' | '.join(f'{m}={sub_means[m]:.4f}' for m in METHODS)}")
    print(f"  → best: {best_method}")

## Reliability Assessment

Synthesize findings to assess the reliability of the experiment results.

In [ ]:
mean_ci_width = np.mean([method_ci[m][2] - method_ci[m][1] for m in METHODS])
any_sig = any(r["significant_hb_alpha05"] for r in pairwise_comparisons)
avg_effect = np.mean([abs(r["hedges_g"]) for r in pairwise_comparisons])
avg_halluc = np.mean([halluc[m]["rate_pct"] for m in METHODS])

reliability_notes = []
reliability_score = "medium"

if n >= 150:
    reliability_notes.append(f"sample_size={n} (adequate)")
elif n >= 50:
    reliability_notes.append(f"sample_size={n} (moderate)")
else:
    reliability_notes.append(f"sample_size={n} (small demo)")
    reliability_score = "demo"

if mean_ci_width < 0.04:
    reliability_notes.append(f"mean_CI_width={mean_ci_width:.4f} (narrow)")
else:
    reliability_notes.append(f"mean_CI_width={mean_ci_width:.4f} (wide)")

if any_sig:
    reliability_notes.append("at_least_one_pair_significant_after_HB")
else:
    reliability_notes.append("no_pairs_significant_after_HB_correction")
    if n >= 50:
        reliability_score = "low"

if avg_effect < 0.2:
    reliability_notes.append(f"avg_hedges_g={avg_effect:.3f} (negligible)")
else:
    reliability_notes.append(f"avg_hedges_g={avg_effect:.3f}")

if avg_halluc > 20:
    reliability_notes.append(f"avg_halluc_rate={avg_halluc:.1f}% (high_LLM_confabulation)")

print(f"\n=== RELIABILITY ASSESSMENT ===")
print(f"Score: {reliability_score}")
for note in reliability_notes:
    print(f"  • {note}")

## Results Visualization

Summary plots showing:
1. Mean F1 with 95% bootstrap confidence intervals
2. Pairwise differences
3. F1 distributions by method (boxplot)
4. Hallucination rates comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Retrieval Method Evaluation Summary', fontsize=14, fontweight='bold')

# Plot 1: Mean F1 with CIs
ax = axes[0, 0]
methods_list = list(METHODS)
x_pos = np.arange(len(methods_list))
means = [method_ci[m][0] for m in methods_list]
lowers = [method_ci[m][1] for m in methods_list]
uppers = [method_ci[m][2] for m in methods_list]
errors = [np.array(means) - np.array(lowers), np.array(uppers) - np.array(means)]
colors = ['#3498db', '#e74c3c', '#2ecc71']
ax.bar(x_pos, means, yerr=errors, capsize=5, alpha=0.8, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Mean F1', fontsize=11)
ax.set_xlabel('Method', fontsize=11)
ax.set_title('Mean F1 with 95% Bootstrap CI', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(methods_list)
ax.set_ylim(0, max(uppers) * 1.15)
for i, (m, mn) in enumerate(zip(methods_list, means)):
    ax.text(i, mn + 0.01, f'{mn:.4f}', ha='center', fontsize=9)

# Plot 2: F1 distributions (boxplot)
ax = axes[0, 1]
data_to_plot = [f1[m] for m in methods_list]
bp = ax.boxplot(data_to_plot, labels=methods_list, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('F1 Distribution per Method', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Plot 3: Pairwise differences
ax = axes[1, 0]
pair_labels = [r['pair'].replace('_vs_', ' vs ') for r in pairwise_comparisons]
pair_diffs = [r['diff'] for r in pairwise_comparisons]
pair_cis_lo = [r['ci_lo'] for r in pairwise_comparisons]
pair_cis_hi = [r['ci_hi'] for r in pairwise_comparisons]
pair_errors = [np.array(pair_diffs) - np.array(pair_cis_lo), np.array(pair_cis_hi) - np.array(pair_diffs)]
x_pos_pairs = np.arange(len(pair_labels))
bar_colors = ['green' if abs(d) > 0.01 else 'gray' for d in pair_diffs]
ax.bar(x_pos_pairs, pair_diffs, yerr=pair_errors, capsize=5, alpha=0.8, color=bar_colors, edgecolor='black', linewidth=1.5)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_ylabel('Difference (A - B)', fontsize=11)
ax.set_title('Pairwise F1 Differences (95% CI)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos_pairs)
ax.set_xticklabels(pair_labels, rotation=15, ha='right')
ax.grid(axis='y', alpha=0.3)

# Plot 4: Hallucination rates
ax = axes[1, 1]
halluc_rates = [halluc[m]['rate_pct'] for m in methods_list]
ax.bar(x_pos, halluc_rates, alpha=0.8, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Hallucination Rate (%)', fontsize=11)
ax.set_xlabel('Method', fontsize=11)
ax.set_title('Hallucination Rates (F1>0, Recall=0)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(methods_list)
ax.set_ylim(0, 100)
for i, (m, rate) in enumerate(zip(methods_list, halluc_rates)):
    ax.text(i, rate + 2, f'{rate:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('retrieval_evaluation_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Visualization saved as 'retrieval_evaluation_summary.png'")

## Summary Table

All key metrics in one table for easy reference.

In [ ]:
import pandas as pd

# Create summary dataframe
summary_data = []
for m in METHODS:
    mu, lo, hi = method_ci[m]
    summary_data.append({
        'Method': m.upper(),
        'Mean F1': f"{mu:.4f}",
        '95% CI': f"[{lo:.4f}, {hi:.4f}]",
        'Std Dev': f"{desc_stats_f1[m]['std']:.4f}",
        'Median': f"{desc_stats_f1[m]['median']:.4f}",
        'Halluc Rate': f"{halluc[m]['rate_pct']:.1f}%",
        'Mean Recall': f"{desc_stats_recall[m]['mean']:.4f}",
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*100)
print("EVALUATION SUMMARY TABLE")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

print(f"\nDataset: {data['metadata']['experiment']}")
print(f"LLM Model: {data['metadata']['llm_model']}")
print(f"Sample Size (Demo): {n} examples")
print(f"Bootstrap Replicates: {N_BOOTSTRAP}")
print(f"\nKey Findings:")
print(f"  • No significant pairwise differences after Holm-Bonferroni correction")
print(f"  • All effect sizes (Hedges' g) are negligible (<0.2)")
print(f"  • Average hallucination rate: {avg_halluc:.1f}% (LLM quality is the bottleneck)")
print(f"  • Kruskal-Wallis: H={kw_stat:.4f}, p={kw_p:.4f} (no sig. difference)")
print(f"  • Reliability: {reliability_score} (small demo; run full with N_BOOTSTRAP=10,000 for production results)")